In [3]:
%load_ext autoreload
%autoreload 2

In [6]:
import sys


import pandas as pd
import matplotlib.pyplot as plt

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_dir / "src"))
from pathlib import Path
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
db_path = project_dir / "data/raw/Final_Database.csv"

import prepare

In [7]:
original = pd.read_csv(db_path, dtype=str, keep_default_na=False)

In [8]:
main_bill_df = original.copy()

In [10]:
main_bill_df["Bill_Text_clean"] = main_bill_df["Bill_Text"].map(prepare.clean_text)

In [11]:
main_bill_df = main_bill_df.drop_duplicates(subset=['Bill_Text_clean'], keep='last')

display(pd.DataFrame({
    "Check": ["Source records", "Original records", "Original fields", "Prepared fields"],
    "Result": [len(main_bill_df), len(original), len(original.columns), len(main_bill_df.columns)],
}))

,Check,Result
0,Source records,2567
1,Original records,2571
2,Original fields,29
3,Prepared fields,30


In [12]:
main_bill_df.info() 

<class 'pandas.DataFrame'>
Index: 2567 entries, 0 to 2570
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Bill_Number                    2567 non-null   str  
 1   measure_code                   2567 non-null   str  
 2   Bill_Title                     2567 non-null   str  
 3   Jurisdiction                   2567 non-null   str  
 4   NCSL_Summary                   2567 non-null   str  
 5   NCSL_Category                  2567 non-null   str  
 6   Bill_URL                       2567 non-null   str  
 7   Version_Date                   2567 non-null   str  
 8   Year                           2567 non-null   str  
 9   Bill_Status                    2567 non-null   str  
 10  State_Political_Control        2567 non-null   str  
 11  Regional_Cluster_ID            2567 non-null   str  
 12  Bill_Text                      2567 non-null   str  
 13  Sponsor                        256

In [14]:
type(main_bill_df["Sponsor_Party"].dropna().iloc[0])    

str

In [15]:
## Make sure the Sponsor_Party column is a list, even if it is stored as a string representation of a list

import ast

def parse_sponsor_parties(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

In [16]:
main_bill_df["Sponsor_Party"] = main_bill_df["Sponsor_Party"].apply(
    parse_sponsor_parties
)

In [17]:
main_bill_df["Sponsor_Party"] = main_bill_df["Sponsor_Party"].apply(
    parse_sponsor_parties
)

In [18]:
def count_parties(parties):
    if not isinstance(parties, list):
        return [0, 0, 0, 0]

    parties = [
        str(p).strip().upper() if p is not None else "UNKNOWN"
        for p in parties
    ]

    d_count = parties.count("D")
    r_count = parties.count("R")
    unknown_count = sum(
        p in ["UNKNOWN", "U", "", "NONE", "NAN"]
        for p in parties
    )
    other_count = len(parties) - d_count - r_count - unknown_count

    return [d_count, r_count, other_count, unknown_count]


In [19]:
columns = [
    "D_Count", "R_Count", "Other_Count", "Unknown_Count"
]

main_bill_df[columns] = pd.DataFrame(
    main_bill_df["Sponsor_Party"].apply(count_parties).tolist(),
    index=main_bill_df.index,
    columns=columns
)

In [20]:
main_bill_df["Total_Sponsors"] = (
    main_bill_df["D_Count"]
    + main_bill_df["R_Count"]
    + main_bill_df["Other_Count"]
    + main_bill_df["Unknown_Count"]
)

main_bill_df["Bipartisan_Support"] = (
    (main_bill_df["D_Count"] > 0)
    & (main_bill_df["R_Count"] > 0)
).astype(int)

main_bill_df["Democrat_Sponsor_Ratio"] = (
    main_bill_df["D_Count"]
    / main_bill_df["Total_Sponsors"].replace(0, float("nan"))
)

main_bill_df["Republican_Sponsor_Ratio"] = (
    main_bill_df["R_Count"]
    / main_bill_df["Total_Sponsors"].replace(0, float("nan"))
)


In [21]:
main_bill_df[
    ["Bill_Number", "Sponsor_Party",
     "D_Count", "R_Count", "Other_Count","Unknown_Count", "Bipartisan_Support", "Total_Sponsors", "Democrat_Sponsor_Ratio", "Republican_Sponsor_Ratio"]
].head(10)

,Bill_Number,Sponsor_Party,D_Count,R_Count,Other_Count,Unknown_Count,Bipartisan_Support,Total_Sponsors,Democrat_Sponsor_Ratio,Republican_Sponsor_Ratio
0,H 208,[R],0,1,0,0,0,1,0.000000,1.0
1,H 365,[R],0,1,0,0,0,1,0.000000,1.0
2,H 515,[R],0,1,0,0,0,1,0.000000,1.0
3,H 516,[D],1,0,0,0,0,1,1.000000,0.0
4,H 557,[D],1,0,0,0,0,1,1.000000,0.0
5,HR 244,[R],0,1,0,0,0,1,0.000000,1.0
6,S 294,[R],0,1,0,0,0,1,0.000000,1.0
7,HCR 3,[R],0,1,0,0,0,1,0.000000,1.0
8,S 2,"[I, D, I]",1,0,2,0,0,3,0.333333,0.0
9,S 33,[],0,0,0,1,0,1,0.000000,0.0


Code above to be added to preprare.py file to serve as part of the common data file for the team. 